# plotlet in a Jupyter notebook

plotlet is a small Python library that emits matplotlib-style SVG plots. Artist methods (`line`, `scatter`, `title`, ...) only *record* what to draw; the SVG is emitted when the chart auto-renders as a cell's last expression or you call `.show()`.

In [1]:
import math
import random
import numpy as np
import pandas as pd

import plotlet as pt

xs = [i * 0.1 for i in range(64)]   # reused below

## Hello world: line plot

The chart auto-renders when it's the cell's last expression — same `_repr_html_` protocol as plotly or pandas DataFrames. Use `.show()` if you need to render mid-cell.

In [2]:
df = {
    "time":  xs,
    "sin_x": [math.sin(t) for t in xs],
    "cos_x": [math.cos(t) for t in xs],
}

c = pt.chart(df, title="Hello from plotlet", xlabel="x", ylabel="y",
             legend=True, grid=True)
c.line(x="time", y="sin_x", label="sin(x)")
c.line(x="time", y="cos_x", label="cos(x)", linestyle="--")

## Bar charts

In [3]:
df = {"category": ["A", "B", "C", "D"], "count": [4, 7, 2, 9]}

c = pt.chart(df, data_width=500, data_height=300, title="bar chart", ylabel="count")
c.bar(x="category", y="count", color="C2")

## Scatter

Lists, numpy arrays, and pandas Series are all accepted (anything with `.tolist()` auto-converts). `color=` splits into colored series.

In [4]:
rng = np.random.default_rng(0)
x_arr = np.linspace(0, 6, 60)
df = pd.DataFrame({
    "x":      np.concatenate([x_arr, x_arr]),
    "y":      np.concatenate([np.sin(x_arr), np.cos(x_arr)]) + rng.normal(0, 0.15, 120),
    "series": ["sin"] * len(x_arr) + ["cos"] * len(x_arr),
})

c = pt.chart(df, xlabel="x", ylabel="y", legend=True, grid=True)
c.scatter(x="x", y="y", color="series", size=2.5, alpha=0.6)

## Categorical axes

Pass strings on either axis and the scale auto-switches to categorical, alphabetical by default. When alphabetical is wrong — for example, doses go `control < low < med < high` but alphabetical scrambles to `control, high, low, med` — use `xscale("category", order=[...])` to lock in a layout. `yscale("category", order=[...])` works the same way.

In [5]:
rng = random.Random(0)
groups = ["control", "low", "med", "high"]
n = 15
df = {
    "dose":  [g for g in groups for _ in range(n)],
    "value": [rng.gauss(i * 0.6, 0.4) for i in range(len(groups)) for _ in range(n)],
}

c = pt.chart(df, data_width=500, data_height=320,
             title="dose response", xlabel="dose", ylabel="value")
c.xscale("category", order=groups)   # without this, the axis would still be categorical but ordered alphabetically
c.scatter(x="dose", y="value", alpha=0.6)

## Histogram

In [6]:
rng = random.Random(0)
df = {"value": [rng.gauss(0, 1) for _ in range(2000)]}

c = pt.chart(df, title="normal(0, 1)")
c.hist(x="value", bins=30, color="C3")

## Confidence bands (`fill_between`)

In [7]:
mean  = [math.sin(t) for t in xs]
lower = [m - 0.25 for m in mean]
upper = [m + 0.25 for m in mean]

c = pt.chart({"x": xs, "mean": mean, "lo": lower, "hi": upper},
             xlabel="x", ylabel="y", legend=True)
c.fill_between(x="x", y1="lo", y2="hi", fill="C0", alpha=0.2, label="\u00b10.25")
c.line(x="x", y="mean", label="sin(x)")

## Reference lines and shaded regions

`axhline` / `axvline` draw infinite reference lines; `axhspan` / `axvspan` draw shaded bands. Neither affects autoscale.

In [8]:
df = {"x": xs, "y": [math.sin(t) for t in xs]}

c = pt.chart(df, xlabel="x", ylabel="y", legend=True)
c.axhspan(-0.5, 0.5, color="C0", alpha=0.1)
c.axvspan(math.pi, 2 * math.pi, color="C2", alpha=0.1)
c.line(x="x", y="y", label="sin(x)")
c.axhline(0, linestyle="--", color="gray")
c.axvline(math.pi, linestyle=":", color="C3", label="x = \u03c0")

## Heatmaps (`imshow`)

Pass any 2-D array; `cmap=` selects from the 180 vendored matplotlib colormaps.

In [9]:
y, x = np.mgrid[-3:3:60j, -3:3:60j]
z = np.exp(-(x**2 + y**2) / 2) * np.cos(3 * x) * np.cos(3 * y)

c = pt.chart(title="2-D field", xlabel="x", ylabel="y")
c.imshow(z, extent=[-3, 3, -3, 3], cmap="viridis")